## 1. torch_prep_kfold.py --initial_split  → writes *_trn_final.csv, *_tst_preprocess.csv

Typical Workflow


Step 1: Initial split (from raw features + labels)
python torch_prep_kfold.py \
  --initial_split \
  --model_type reg \
  --prefix gbsa \
  --ref_file ref.csv \
  --ref_id_col sequence \
  --ref_label_col bind_avg \
  --filenames features1.csv features2.csv \
  --feature_id_col sequence \
  --test_percentage 0.15 \
  --scramble_fractions 0.0


In [1]:
import sys
import logging
import argparse
import numpy as np
import pandas as pd
from typing import Any, Optional, List, Tuple
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
import os

In [7]:
###############################################################################
# Data Loading (Reference + Features)
###############################################################################

In [ ]:
id_col = 'sequence'
label_col = 'bind_avg'
df1 = pd.read_csv('exp_data_all.csv')
ref_data = df1[[id_col, label_col]].copy()

print(ref_data)

                                 sequence  bind_avg
0    GTACACAATTTTTTACAAAATTTAAATTAAAACAAA -0.862667
1    GCAGCCGAGGCGGAGAGAGAGAGAGGACAGCTTACG -0.703319
2    AGGCCCAGGAAGAACAATGGCTCTGCCAACTGGGCA -0.659464
3    CTTCCTCACCTGCAGACTTCCTTCCCTGAGTCCCAG -0.543823
4    TGAGGGTCAGAGGCACCCCTTCCTGGAATCTCCTTC -0.424828
..                                    ...       ...
163  ATCTCCTGGGGCGACCACGAGGTCACCCGTCCAGGT  1.524243
164  GAAAACCAGCGAGACCGCATGGTCTCACTTATAAGT  1.452711
165  AGGGAGTTCTCACACCATGTGGGTGGGATTGTAACT  1.305160
166  ACACTGAGCTTCCTCCACGTGCCCAGGTCCTGGCAG  1.430431
167  GTGTCTCCATTGGGGCACGTGTTTATATGTTTATAA  1.598717

[168 rows x 2 columns]


In [6]:
usecols = ['sequence','run','VDWAALS','EEL','EGB','ESURF','HB Energy','Hydrophobic Energy','Pi-Pi Energy','Delta_Entropy']

df2 = pd.read_csv('rawdat.csv', usecols=usecols)
feature_data = df2.copy()

print(feature_data.head())

                               sequence  run  VDWAALS       EEL       EGB  \
0  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -252.110 -1886.830  1841.253   
1  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -238.510 -1881.424  1835.847   
2  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -246.721 -1895.687  1851.589   
3  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -235.671 -1857.573  1814.002   
4  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -230.214 -1897.268  1847.934   

    ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  Delta_Entropy  
0 -36.482  -1.940432         -165.447020 -1.655191e-03     -26.046553  
1 -36.023  -2.003962         -155.422935 -4.708262e-02     -24.150637  
2 -35.802  -2.269901         -142.386371 -5.901517e-29     -24.329875  
3 -34.799  -2.838678         -147.918585 -3.236084e-07     -23.615145  
4 -34.391  -2.810414         -151.012478 -1.784784e-05     -23.698348  


In [8]:
## I am not doing sequence level scrambling

1) --initial_split:
   - Merges feature data (optionally from multiple files) with reference data using a shared ID column.
   - Performs an optional "sequence-level scrambling" of labels in the TRAINING set only, controlled by `scramble_fractions`.
   - Splits into train and test sets by unique sequence ID (or stratified for classification).
   - Saves the resulting CSV files:
       * PREFIX_MODELTYPE_scrFRAC_trn_final.csv   (training)
       * PREFIXMODELTYPE_scrFRAC_tst_preprocess.csv (test)